In [ ]:
from pathlib import Path
import sys
import copy
from prettytable import PrettyTable

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from rl.algos.d3qn.trainer import load_config, train, evaluate
from rl.utils.path import build_run_name, build_run_paths
from rl.utils.plotting import print_stats, plot_multiple_conf_interval

base_config = load_config(ROOT / 'configs' / 'default.yaml')
base_config.train.num_episodes = 10
base_config.env.trading_period = 500
base_config.env.train_split = 0.8


In [ ]:
N_TEST = 5

profit_config = copy.deepcopy(base_config)
profit_config.env.reward = 'profit'
profit_run_name = build_run_name('profit_d3qn')
profit_paths = build_run_paths(ROOT / 'runs', profit_run_name)
profit_paths.run_dir.mkdir(parents=True, exist_ok=True)
train(profit_config, profit_paths)
profit_ckpt = profit_paths.checkpoints_dir / 'checkpoint_latest.pt'
_, _, profit_returns = evaluate(
    profit_config,
    checkpoint_path=profit_ckpt,
    episodes=N_TEST,
    epsilon=profit_config.train.eval_epsilon,
    device=profit_config.run.device,
)

sharpe_config = copy.deepcopy(base_config)
sharpe_config.env.reward = 'sr'
sharpe_run_name = build_run_name('sharpe_d3qn')
sharpe_paths = build_run_paths(ROOT / 'runs', sharpe_run_name)
sharpe_paths.run_dir.mkdir(parents=True, exist_ok=True)
train(sharpe_config, sharpe_paths)
sharpe_ckpt = sharpe_paths.checkpoints_dir / 'checkpoint_latest.pt'
_, _, sharpe_returns = evaluate(
    sharpe_config,
    checkpoint_path=sharpe_ckpt,
    episodes=N_TEST,
    epsilon=sharpe_config.train.eval_epsilon,
    device=sharpe_config.run.device,
)

table = PrettyTable(['Trading System', 'Avg. Return (%)', 'Max Return (%)', 'Min Return (%)', 'Std. Dev.'])
print_stats('Profit D3QN', profit_returns, table)
print_stats('Sharpe D3QN', sharpe_returns, table)
print(table)

plot_multiple_conf_interval(['Profit D3QN', 'Sharpe D3QN'], [profit_returns, sharpe_returns])
